# ATML PA3 - Task 1: Large Language Model Fine-Tuning
## Kaggle Execution Notebook

This notebook runs the complete end-to-end pipeline for **Task 1** of EE-5102/CS-6304 (ATML PA3):
1. **MetaMathQA Filtering & Activation Probe Sampling** (seed 6304, GSM8K test exclusion, max 2048 tokens)
2. **Shared Baseline Evaluation** (`Qwen/Qwen3-0.6B` zero-shot on GSM8K, HellaSwag, BoolQ, ARC-Easy)
3. **Five Training Variants**:
   - Full Fine-Tuning (Full FT, $\lambda=0$)
   - Full FT + L2-SP ($\lambda=10^{-3}$)
   - LoRA-4 ($r=4, \alpha=8$)
   - LoRA-16 ($r=16, \alpha=32$)
   - OPLoRA-16 ($r=16, \alpha=32, k=128$, with frozen singular bases $U_k, V_k$)
4. **Zero-Shot Benchmark Evaluation** across all 7 conditions via `lm-eval` Python API
5. **Weight-Space & Activation Diagnostics** on `down_proj` in layers 0, 13, and 27
6. **Qualitative Evaluation & 42-Response Annotation** (`first_plot_questions.yaml`)
7. **Report Artifact Generation & Output Zip Packaging**

### Step 0: Hardware Verification & Repository Setup

In [1]:
# Verify GPU
!nvidia-smi

import os
import sys

REPO_URL = "https://github.com/ahmad-sarfraz345/ATML-PA3.git"
REPO_DIR = "/kaggle/working/ATML-PA3"

if not os.path.exists(REPO_DIR):
    print("Cloning repository...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Updating repository...")
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
sys.path.append(REPO_DIR)
print("Working directory:", os.getcwd())

Tue Nov  3 14:22:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 1: Install Environment Dependencies

In [2]:
!pip install -q --upgrade pip
!pip install -q -r requirements.txt
print("Dependencies successfully installed.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Dependencies successfully installed.


### Step 2: Modular Stage Controls
Configure which stages to execute in the current session. All checkpoints and results persist in `outputs/task1`.

In [3]:
# Stage execution toggles
STAGE_PREPARE_DATA       = True
STAGE_EVAL_BASELINE      = True
STAGE_TRAIN_FULL_FT      = True
STAGE_TRAIN_FULL_FT_L2SP = True
STAGE_TRAIN_LORA_4       = True
STAGE_TRAIN_LORA_16      = True
STAGE_TRAIN_OPLORA_16    = True
STAGE_EVAL_BENCHMARKS    = True
STAGE_DIAGNOSTICS        = True
STAGE_QUALITATIVE        = True
STAGE_ANALYSIS           = True

MODEL_ID = "Qwen/Qwen3-0.6B"
SEED = 6304

### Step 3: Prepare MetaMathQA 40k & Sample Activation Probe Set

In [4]:
if STAGE_PREPARE_DATA:
    print("--- Executing Stage: Prepare Data ---")
    !python -m scripts.run_task1 --stage prepare_data --model_name_or_path {MODEL_ID} --seed {SEED}

--- Executing Stage: Prepare Data ---
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/ATML-PA3/scripts/run_task1.py", line 12, in <module>
    from task1.qualitative import run_qualitative_generation
  File "/kaggle/working/ATML-PA3/task1/qualitative.py", line 102, in <module>
    ) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
         ^^^^^
NameError: name 'Tuple' is not defined. Did you mean: 'tuple'?


### Step 4: Evaluate Baseline (Unchanged Model, 0 Updates)

In [5]:
if STAGE_EVAL_BASELINE:
    print("--- Executing Stage: Evaluate Baseline ---")
    !python -m scripts.run_task1 --stage eval_baseline --model_name_or_path {MODEL_ID} --seed {SEED}

--- Executing Stage: Evaluate Baseline ---
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/ATML-PA3/scripts/run_task1.py", line 12, in <module>
    from task1.qualitative import run_qualitative_generation
  File "/kaggle/working/ATML-PA3/task1/qualitative.py", line 102, in <module>
    ) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
         ^^^^^
NameError: name 'Tuple' is not defined. Did you mean: 'tuple'?


### Step 5: Train Model Variants
1. Full Fine-Tuning ($lr=2\times 10^{-5}, \lambda=0$)
2. Full FT + L2-SP ($lr=2\times 10^{-5}, \lambda=10^{-3}$)
3. LoRA Rank 4 ($r=4, \alpha=8, lr=2\times 10^{-4}$)
4. LoRA Rank 16 ($r=16, \alpha=32, lr=2\times 10^{-4}$)
5. OPLoRA Rank 16 ($r=16, \alpha=32, k=128, lr=2\times 10^{-4}$)

In [6]:
if STAGE_TRAIN_FULL_FT:
    print("--- Executing Stage: Train Full FT ---")
    !python -m scripts.run_task1 --stage train_full_ft --model_name_or_path {MODEL_ID} --seed {SEED}

if STAGE_TRAIN_FULL_FT_L2SP:
    print("--- Executing Stage: Train Full FT + L2-SP ---")
    !python -m scripts.run_task1 --stage train_full_ft_l2sp --model_name_or_path {MODEL_ID} --seed {SEED}

if STAGE_TRAIN_LORA_4:
    print("--- Executing Stage: Train LoRA-4 ---")
    !python -m scripts.run_task1 --stage train_lora_4 --model_name_or_path {MODEL_ID} --seed {SEED}

if STAGE_TRAIN_LORA_16:
    print("--- Executing Stage: Train LoRA-16 ---")
    !python -m scripts.run_task1 --stage train_lora_16 --model_name_or_path {MODEL_ID} --seed {SEED}

if STAGE_TRAIN_OPLORA_16:
    print("--- Executing Stage: Train OPLoRA-16 ---")
    !python -m scripts.run_task1 --stage train_oplora_16 --model_name_or_path {MODEL_ID} --seed {SEED}

--- Executing Stage: Train Full FT ---
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/ATML-PA3/scripts/run_task1.py", line 12, in <module>
    from task1.qualitative import run_qualitative_generation
  File "/kaggle/working/ATML-PA3/task1/qualitative.py", line 102, in <module>
    ) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
         ^^^^^
NameError: name 'Tuple' is not defined. Did you mean: 'tuple'?
--- Executing Stage: Train Full FT + L2-SP ---
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/ATML-PA3/scripts/run_task1.py", line 12, in <module>
    from task1.qualitative import run_qualitative_generation
  File "/kaggle/working/ATML-PA3/task1/qualitative.py", line 102, in <module>
    ) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    

### Step 6: Zero-Shot Benchmark Evaluation (7 Conditions x 4 Tasks)

In [7]:
if STAGE_EVAL_BENCHMARKS:
    print("--- Executing Stage: Evaluate Checkpoints on Benchmarks ---")
    !python -m scripts.run_task1 --stage eval_benchmarks --model_name_or_path {MODEL_ID} --seed {SEED}

--- Executing Stage: Evaluate Checkpoints on Benchmarks ---
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/ATML-PA3/scripts/run_task1.py", line 12, in <module>
    from task1.qualitative import run_qualitative_generation
  File "/kaggle/working/ATML-PA3/task1/qualitative.py", line 102, in <module>
    ) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
         ^^^^^
NameError: name 'Tuple' is not defined. Did you mean: 'tuple'?


### Step 7: Weight-Space & Activation Diagnostics

In [8]:
if STAGE_DIAGNOSTICS:
    print("--- Executing Stage: Diagnostics ---")
    !python -m scripts.run_task1 --stage diagnostics --model_name_or_path {MODEL_ID} --seed {SEED}

--- Executing Stage: Diagnostics ---
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/ATML-PA3/scripts/run_task1.py", line 12, in <module>
    from task1.qualitative import run_qualitative_generation
  File "/kaggle/working/ATML-PA3/task1/qualitative.py", line 102, in <module>
    ) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
         ^^^^^
NameError: name 'Tuple' is not defined. Did you mean: 'tuple'?


### Step 8: Qualitative Generations (168 Total, 42 Annotated)

In [9]:
if STAGE_QUALITATIVE:
    print("--- Executing Stage: Qualitative Generations ---")
    !python -m scripts.run_task1 --stage qualitative --model_name_or_path {MODEL_ID} --seed {SEED}

--- Executing Stage: Qualitative Generations ---
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/ATML-PA3/scripts/run_task1.py", line 12, in <module>
    from task1.qualitative import run_qualitative_generation
  File "/kaggle/working/ATML-PA3/task1/qualitative.py", line 102, in <module>
    ) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
         ^^^^^
NameError: name 'Tuple' is not defined. Did you mean: 'tuple'?


### Step 9: Synthesize Results, Figures, and Tables

In [10]:
if STAGE_ANALYSIS:
    print("--- Executing Stage: Analysis & Plots ---")
    !python -m scripts.run_task1 --stage analysis --model_name_or_path {MODEL_ID} --seed {SEED}

from IPython.display import Image, display, Markdown
import pandas as pd

# Display Learning Curves
if os.path.exists("outputs/task1/learning_curves.png"):
    display(Image("outputs/task1/learning_curves.png"))

# Display Benchmark Summary Table
if os.path.exists("outputs/task1/benchmark_summary.md"):
    with open("outputs/task1/benchmark_summary.md", "r") as f:
        display(Markdown(f.read()))

# Display Resource Usage Summary
if os.path.exists("outputs/task1/resource_usage_summary.md"):
    with open("outputs/task1/resource_usage_summary.md", "r") as f:
        display(Markdown(f.read()))

--- Executing Stage: Analysis & Plots ---
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/ATML-PA3/scripts/run_task1.py", line 12, in <module>
    from task1.qualitative import run_qualitative_generation
  File "/kaggle/working/ATML-PA3/task1/qualitative.py", line 102, in <module>
    ) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
         ^^^^^
NameError: name 'Tuple' is not defined. Did you mean: 'tuple'?


### Step 10: Package All Outputs into a Downloadable Zip Archive

In [11]:
import shutil
from pathlib import Path

zip_filename = "/kaggle/working/task1_outputs"
source_dir = "outputs/task1"

print(f"Archiving {source_dir} to {zip_filename}.zip ...")
shutil.make_archive(zip_filename, 'zip', source_dir)
print(f"SUCCESS: Task 1 outputs packaged at {zip_filename}.zip")
print(f"Size: {Path(zip_filename + '.zip').stat().st_size / (1024*1024):.2f} MB")

Archiving outputs/task1 to /kaggle/working/task1_outputs.zip ...


FileNotFoundError: [Errno 2] No such file or directory: 'outputs/task1'